# 05 · Spark SQL — DataFrame API vs SQL puro

Mismo problema resuelto de dos formas, para comprobar si Catalyst genera el mismo plan de ejecucion aunque cambie la sintaxis. Reutilizamos los datos de Chicago Crimes del proyecto 04.

Cuatro comparaciones, todas con la misma estructura: DataFrame API -> `explain()`, Spark SQL -> `explain()`, reflexion sobre lo que muestra el plan.

In [1]:
import sys
import re
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontro la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.spark_session import create_spark_session
from pyspark.sql import functions as F

spark = create_spark_session(app_name="05-spark-sql")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/22 11:28:31 WARN Utils: Your hostname, MacBook-Air-de-Gloria.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.20 instead (on interface en0)
26/09/22 11:28:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 11:28:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/22 11:28:35 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Carga de datos

No repetimos el pipeline Silver completo del proyecto 04 (profiling, geolocalizacion, deduplicacion) — aqui no es el objetivo. Cargamos el mismo CSV con el mismo schema explicito para evitar la inferencia, normalizamos los nombres de columna a snake_case y casteamos `date`. Con eso alcanza para las 4 comparaciones.

In [2]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, BooleanType, DoubleType
)

DATASETS_PATH = PROJECT_ROOT / "datasets" / "04-chicago-crimes"
file_path = DATASETS_PATH / "Crimes_-_2001_to_Present_20260727.csv"

schema = StructType([
    StructField("ID", IntegerType(), True),
    StructField("Case Number", StringType(), True),
    StructField("Date", StringType(), True),
    StructField("Block", StringType(), True),
    StructField("IUCR", StringType(), True),
    StructField("Primary Type", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Location Description", StringType(), True),
    StructField("Arrest", BooleanType(), True),
    StructField("Domestic", BooleanType(), True),
    StructField("Beat", IntegerType(), True),
    StructField("District", IntegerType(), True),
    StructField("Ward", IntegerType(), True),
    StructField("Community Area", IntegerType(), True),
    StructField("FBI Code", StringType(), True),
    StructField("X Coordinate", IntegerType(), True),
    StructField("Y Coordinate", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Updated On", StringType(), True),
    StructField("Latitude", DoubleType(), True),
    StructField("Longitude", DoubleType(), True),
    StructField("Location", StringType(), True),
])

raw_df = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(file_path))
)

raw_df.count()

8602048

In [3]:
def normalize_column_name(column):
    column = column.strip()
    column = re.sub(r"[^a-zA-Z0-9]+", "_", column)
    return column.strip("_").lower()

df = raw_df.toDF(*[normalize_column_name(c) for c in raw_df.columns])
df = df.withColumn("date", F.to_timestamp("date", "MM/dd/yyyy hh:mm:ss a"))

df.select("id", "date", "year", "primary_type", "district", "arrest").show(5, truncate=False)

+--------+-------------------+----+---------------+--------+------+
|id      |date               |year|primary_type   |district|arrest|
+--------+-------------------+----+---------------+--------+------+
|14273247|2026-07-19 00:00:00|2026|CRIMINAL DAMAGE|16      |false |
|14269780|2026-07-19 00:00:00|2026|BATTERY        |8       |false |
|14266825|2026-07-19 00:00:00|2026|THEFT          |12      |false |
|14267074|2026-07-19 00:00:00|2026|THEFT          |6       |false |
|14266739|2026-07-19 00:00:00|2026|CRIMINAL DAMAGE|6       |true  |
+--------+-------------------+----+---------------+--------+------+
only showing top 5 rows


In [4]:
df.createOrReplaceTempView("crimes")

spark.sql("SELECT COUNT(*) AS total_rows FROM crimes").show()

+----------+
|total_rows|
+----------+
|   8602048|
+----------+



## 2. Comparacion 1 — Filtro + seleccion

Elijo un año concreto (por ejemplo el ultimo ano completo en el dataset) y selecciona solo unas pocas columnas: `case_number`, `date`, `primary_type`, `district`, `arrest`.

**Con DataFrame API:**

In [ ]:
resultado_df_1 = df.filter(F.col("year") == 2023).select("case_number", "date", "primary_type", "district", "arrest")
resultado_df_1.explain(True)

== Parsed Logical Plan ==
'Project ['case_number, 'date, 'primary_type, 'district, 'arrest]
+- Filter (year#65 = 2023)
   +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y_coordinate#64, year#65, updated_on#66, latitude#67, longitude#68, location#69]
      +- Project [ID#0 AS id#48, Case Number#1 AS case_number#49, Date#2 AS date#50, Block#3 AS block#51, IUCR#4 AS iucr#52, Primary Type#5 AS primary_type#53, Description#6 AS description#54, Location Description#7 AS location_description#55, Arrest#8 AS arrest#56, Domestic#9 AS domestic#57, Beat#10 AS beat#58, District#11 AS district#59, Ward#12 AS ward#60, Community Area#13 AS community_area#61, FBI Code#14 AS fbi_code#62, X Coordinate#15 AS x_coordinate#63, Y Coordinate#16 AS y

**Con Spark SQL, sobre la vista `crimes`:**

In [ ]:
resultado_sql_1 = spark.sql("""
    SELECT case_number, date, primary_type, district, arrest
    FROM crimes
    WHERE year = 2023
""")

resultado_sql_1.explain(True)

== Parsed Logical Plan ==
'Project ['case_number, 'date, 'primary_type, 'district, 'arrest]
+- 'Filter ('year = 2023)
   +- 'UnresolvedRelation [crimes], [], false

== Analyzed Logical Plan ==
case_number: string, date: timestamp, primary_type: string, district: int, arrest: boolean
Project [case_number#49, date#71, primary_type#53, district#59, arrest#56]
+- Filter (year#65 = 2023)
   +- SubqueryAlias crimes
      +- View (`crimes`, [id#48, case_number#49, date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y_coordinate#64, year#65, updated_on#66, latitude#67, longitude#68, location#69])
         +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward

<div style="background:#242938;border-left:5px solid #34d399;border-radius:8px;padding:14px 20px;color:#dcdfea;">

<p style="margin:0 0 10px 0;"><b>Reflexión — comparación 1</b> &nbsp; <span style="background:#103b2c;color:#6ee7b7;padding:2px 10px;border-radius:999px;font-size:0.78em;font-weight:600;">✅ Planes idénticos</span></p>

<ul style="margin:0 0 10px 0;padding-left:20px;line-height:1.7;">
<li>El número de filas coincide en las dos versiones (mismo resultado).</li>
<li>El Optimized Logical Plan es idéntico carácter a carácter en ambas: Project sobre Filter (isnotnull(Year) AND Year = 2023) sobre la Relation del CSV. Catalyst añadió el isnotnull(Year) por su cuenta, no lo escribí yo.</li>
<li>El Physical Plan también es idéntico: el filtro llega empujado hasta el propio FileScan (PushedFilters) — predicate pushdown real, no solo teórico.</li>
</ul>

<p style="margin:0;color:#dcdfea;">💡 <b>Conclusión:</b> para una consulta de filtro + selección, DataFrame API y SQL no solo dan el mismo resultado, generan literalmente el mismo plan de ejecución.</p>

</div>

## 3. Comparacion 2 — Agregacion

Numero de delitos por `primary_type`, ordenado de mayor a menor.

**Con DataFrame API:**

In [7]:
resultado_df_2 = df.groupBy("primary_type").count().orderBy(F.col("count").desc())
resultado_df_2.explain(True)

== Parsed Logical Plan ==
'Sort ['count DESC NULLS LAST], true
+- Aggregate [primary_type#53], [primary_type#53, count(1) AS count#105L]
   +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y_coordinate#64, year#65, updated_on#66, latitude#67, longitude#68, location#69]
      +- Project [ID#0 AS id#48, Case Number#1 AS case_number#49, Date#2 AS date#50, Block#3 AS block#51, IUCR#4 AS iucr#52, Primary Type#5 AS primary_type#53, Description#6 AS description#54, Location Description#7 AS location_description#55, Arrest#8 AS arrest#56, Domestic#9 AS domestic#57, Beat#10 AS beat#58, District#11 AS district#59, Ward#12 AS ward#60, Community Area#13 AS community_area#61, FBI Code#14 AS fbi_code#62, X Coordinate#15 AS x_coordinate#63, Y 

**Con Spark SQL:**

In [ ]:
resultado_sql_2 = spark.sql("""
    SELECT primary_type, 
    COUNT(*) AS count
    FROM crimes
    GROUP BY primary_type
    ORDER BY count DESC
""")

resultado_sql_2.explain(True)

== Parsed Logical Plan ==
'Sort ['count DESC NULLS LAST], true
+- 'Aggregate ['primary_type], ['primary_type, 'COUNT(1) AS count#131]
   +- 'UnresolvedRelation [crimes], [], false

== Analyzed Logical Plan ==
primary_type: string, count: bigint
Sort [count#131L DESC NULLS LAST], true
+- Aggregate [primary_type#53], [primary_type#53, count(1) AS count#131L]
   +- SubqueryAlias crimes
      +- View (`crimes`, [id#48, case_number#49, date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y_coordinate#64, year#65, updated_on#66, latitude#67, longitude#68, location#69])
         +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi

<div style="background:#242938;border-left:5px solid #34d399;border-radius:8px;padding:14px 20px;color:#dcdfea;">

<p style="margin:0 0 10px 0;"><b>Reflexión — comparación 2</b> &nbsp; <span style="background:#103b2c;color:#6ee7b7;padding:2px 10px;border-radius:999px;font-size:0.78em;font-weight:600;">✅ Planes idénticos</span></p>

<ul style="margin:0 0 10px 0;padding-left:20px;line-height:1.7;">
<li>Aparece HashAggregate dos veces: una parcial (partial_count(1), agrega dentro de cada partición) y una final (combina los conteos parciales). Es el patrón estándar de cualquier GROUP BY distribuido.</li>
<li>Hay dos Exchange (shuffles), pero por motivos distintos: el primero (hashpartitioning por primary_type) reparte los datos para poder agregar; el segundo (rangepartitioning por count) es por el ORDER BY, no por el GROUP BY.</li>
<li>El FileScan solo lee la columna Primary Type (ReadSchema: struct&lt;Primary Type:string&gt;) de las 21 que tiene el CSV — column pruning.</li>
<li>Los planes físicos de DataFrame API y SQL son, otra vez, idénticos.</li>
</ul>

<p style="margin:0;color:#dcdfea;">💡 <b>Conclusión:</b> el GROUP BY por sí solo añade 2 HashAggregate + 1 Exchange; el ORDER BY añade su propio Exchange aparte.</p>

</div>

## 4. Comparacion 3 — Agregacion con calculo

Tasa de arrestos (`% arrest = true`) por `primary_type`.

**Con DataFrame API:**

In [9]:
resultado_df_3 = df.groupBy("primary_type") \
    .agg(
        F.count("*").alias("total_count"),
        F.sum(F.col("arrest").cast("int")).alias("arrest_count")
    ) \
    .withColumn("arrest_rate", F.round(F.col("arrest_count") / F.col("total_count"), 2)) \
    .orderBy(F.col("arrest_rate").desc())

resultado_df_3.explain(True)

== Parsed Logical Plan ==
'Sort ['arrest_rate DESC NULLS LAST], true
+- Project [primary_type#53, total_count#135L, arrest_count#136L, round((cast(arrest_count#136L as double) / cast(total_count#135L as double)), 2) AS arrest_rate#161]
   +- Aggregate [primary_type#53], [primary_type#53, count(1) AS total_count#135L, sum(cast(arrest#56 as int)) AS arrest_count#136L]
      +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y_coordinate#64, year#65, updated_on#66, latitude#67, longitude#68, location#69]
         +- Project [ID#0 AS id#48, Case Number#1 AS case_number#49, Date#2 AS date#50, Block#3 AS block#51, IUCR#4 AS iucr#52, Primary Type#5 AS primary_type#53, Description#6 AS description#54, Location Description#7 AS location_de

**Con Spark SQL:**

In [11]:
resultado_sql_3 = spark.sql("""
    SELECT primary_type,
    COUNT(*) AS total_count,
    SUM(CASE WHEN arrest THEN 1 ELSE 0 END) AS arrest_count,
    ROUND(SUM(CASE WHEN arrest THEN 1 ELSE 0 END) / COUNT(*), 2) AS arrest_rate
    FROM crimes
    GROUP BY primary_type
    ORDER BY arrest_rate DESC
""")

resultado_sql_3.explain(True)

== Parsed Logical Plan ==
'Sort ['arrest_rate DESC NULLS LAST], true
+- 'Aggregate ['primary_type], ['primary_type, 'COUNT(1) AS total_count#177, 'SUM(CASE WHEN 'arrest THEN 1 ELSE 0 END) AS arrest_count#178, 'ROUND(('SUM(CASE WHEN 'arrest THEN 1 ELSE 0 END) / 'COUNT(1)), 2) AS arrest_rate#179]
   +- 'UnresolvedRelation [crimes], [], false

== Analyzed Logical Plan ==
primary_type: string, total_count: bigint, arrest_count: bigint, arrest_rate: double
Sort [arrest_rate#179 DESC NULLS LAST], true
+- Aggregate [primary_type#53], [primary_type#53, count(1) AS total_count#177L, sum(CASE WHEN arrest#56 THEN 1 ELSE 0 END) AS arrest_count#178L, round((cast(sum(CASE WHEN arrest#56 THEN 1 ELSE 0 END) as double) / cast(count(1) as double)), 2) AS arrest_rate#179]
   +- SubqueryAlias crimes
      +- View (`crimes`, [id#48, case_number#49, date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, 

<div style="background:#242938;border-left:5px solid #fbbf24;border-radius:8px;padding:14px 20px;color:#dcdfea;">

<p style="margin:0 0 10px 0;"><b>Reflexión — comparación 3</b> &nbsp; <span style="background:#453313;color:#fcd34d;padding:2px 10px;border-radius:999px;font-size:0.78em;font-weight:600;">🟡 Casi idénticos</span></p>

<ul style="margin:0 0 10px 0;padding-left:20px;line-height:1.7;">
<li>La división (arrest_rate) sí aparece como un paso aparte, pero <b>solo en la versión DataFrame API</b>: un Project extra entre el Exchange y el HashAggregate final, porque .withColumn() encadenado tras .agg() queda como una transformación separada que Catalyst no llegó a fusionar del todo.</li>
<li>En la versión SQL, el ROUND(...) escrito directamente dentro del SELECT se integra en la lista de columnas de salida del propio HashAggregate final — sin Project adicional.</li>
<li>Es la única de las 4 comparaciones donde el plan <b>no</b> es 100% idéntico. Aun así, el número de Exchange (shuffles) es el mismo en las dos — el coste real no cambia, es un nodo Project barato de más.</li>
</ul>

<p style="margin:0;color:#dcdfea;">💡 <b>Conclusión:</b> los planes no siempre acaban siendo carácter a carácter iguales, pero cuando difieren en algo tan pequeño como esto, el coste (shuffles) puede seguir siendo idéntico.</p>

</div>

## 5. Comparacion 4 — Consulta mas completa

Top 3 tipos de delito por distrito, usando `Window` + `rank()` en DataFrame API y `RANK() OVER (...)` en SQL — la misma idea de window functions que ya viste en el proyecto 02, aplicada aqui.

**Con DataFrame API:**

In [12]:
from pyspark.sql.window import Window

resultado_df_4 = df.groupBy("district", "primary_type") \
    .count() \
    .withColumn("rank", F.rank().over(Window.partitionBy("district").orderBy(F.col("count").desc()))) \
    .filter(F.col("rank") <= 3) \
    .orderBy("district", "rank")

resultado_df_4.explain(True)

== Parsed Logical Plan ==
'Sort ['district ASC NULLS FIRST, 'rank ASC NULLS FIRST], true
+- Filter (rank#212 <= 3)
   +- Project [district#59, primary_type#53, count#188L, rank#212]
      +- Project [district#59, primary_type#53, count#188L, rank#212, rank#212]
         +- Window [rank(count#188L) windowspecdefinition(district#59, count#188L DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rank#212], [district#59], [count#188L DESC NULLS LAST]
            +- Project [district#59, primary_type#53, count#188L]
               +- Aggregate [district#59, primary_type#53], [district#59, primary_type#53, count(1) AS count#188L]
                  +- Project [id#48, case_number#49, to_timestamp(date#50, Some(MM/dd/yyyy hh:mm:ss a), TimestampType, Some(UTC), true) AS date#71, block#51, iucr#52, primary_type#53, description#54, location_description#55, arrest#56, domestic#57, beat#58, district#59, ward#60, community_area#61, fbi_code#62, x_coordinate#63, y

**Con Spark SQL (usa una CTE con `WITH`):**

In [ ]:
resultado_sql_4 = spark.sql("""
    WITH 
    
    counts AS (
        SELECT district, primary_type, COUNT(*) AS crime_count
        FROM crimes
        GROUP BY district, primary_type
    ),

    ranked AS (
    SELECT *, 
        RANK() OVER (PARTITION BY district ORDER BY crime_count DESC) AS rnk
    FROM counts
    )

    SELECT * 
    FROM ranked
    WHERE rnk <= 3
    ORDER BY district, rnk ASC


""")

resultado_sql_4.explain(True)

== Parsed Logical Plan ==
CTE [counts, ranked]
:  :- 'SubqueryAlias counts
:  :  +- 'Aggregate ['district, 'primary_type], ['district, 'primary_type, 'COUNT(1) AS crime_count#1191]
:  :     +- 'UnresolvedRelation [crimes], [], false
:  +- 'SubqueryAlias ranked
:     +- 'Project [*, 'RANK() windowspecdefinition('district, 'crime_count DESC NULLS LAST, unspecifiedframe$()) AS rnk#1192]
:        +- 'UnresolvedRelation [counts], [], false
+- 'Sort ['district ASC NULLS FIRST, 'rnk ASC NULLS FIRST], true
   +- 'Project [*]
      +- 'Filter ('rnk <= 3)
         +- 'UnresolvedRelation [ranked], [], false

== Analyzed Logical Plan ==
district: int, primary_type: string, crime_count: bigint, rnk: int
WithCTE
:- CTERelationDef 17, false
:  +- SubqueryAlias counts
:     +- Aggregate [district#59, primary_type#53], [district#59, primary_type#53, count(1) AS crime_count#1191L]
:        +- SubqueryAlias crimes
:           +- View (`crimes`, [id#48, case_number#49, date#71, block#51, iucr#52, primary_

In [27]:
resultado_sql_4.show(10, truncate=False)

+--------+-------------------+-----------+---+
|district|primary_type       |crime_count|rnk|
+--------+-------------------+-----------+---+
|NULL    |DECEPTIVE PRACTICE |7          |1  |
|NULL    |BATTERY            |6          |2  |
|NULL    |MOTOR VEHICLE THEFT|5          |3  |
|NULL    |ROBBERY            |5          |3  |
|1       |THEFT              |152506     |1  |
|1       |BATTERY            |39434      |2  |
|1       |DECEPTIVE PRACTICE |35217      |3  |
|2       |THEFT              |82805      |1  |
|2       |BATTERY            |80907      |2  |
|2       |CRIMINAL DAMAGE    |42998      |3  |
+--------+-------------------+-----------+---+
only showing top 10 rows


<div style="background:#242938;border-left:5px solid #34d399;border-radius:8px;padding:14px 20px;color:#dcdfea;">

<p style="margin:0 0 10px 0;"><b>Reflexión — comparación 4</b> &nbsp; <span style="background:#103b2c;color:#6ee7b7;padding:2px 10px;border-radius:999px;font-size:0.78em;font-weight:600;">✅ Planes idénticos (tras el fix)</span></p>

<ul style="margin:0 0 10px 0;padding-left:20px;line-height:1.7;">
<li>Al ejecutarlo la primera vez, el SQL no tenía ORDER BY final y el DataFrame sí — esa diferencia en el plan (un Sort+Exchange de más en la versión DataFrame) no era por la API, era porque las dos consultas no eran realmente equivalentes. Al añadir ORDER BY district, rnk al SQL, los planes físicos pasaron a coincidir operador por operador.</li>
<li>El operador Window es idéntico en las dos versiones, con dos Sort antes de cada WindowGroupLimit (Partial y Final).</li>
<li>La CTE (WITH counts AS (...), ranked AS (...)) se aplana por completo: en el Analyzed Logical Plan aparecen como CTERelationDef separadas, pero en el Optimized Logical Plan ya es una única cadena lineal, sin rastro de que existieran como bloques aparte.</li>
<li>Aparece WindowGroupLimit (Partial + Final) en las dos versiones, es una optimización automática de Catalyst para el patrón "rank + filtro ≤ 3", no algo que pidiera ninguna de las dos APIs.</li>
</ul>

<p style="margin:0;color:#dcdfea;">💡 <b>Conclusión:</b> incluso en la comparación más compleja (window function + CTE), una vez que las dos consultas describen realmente lo mismo, el plan converge. Este patrón se sostiene en las 4 comparaciones realizadas.</p>

</div>

## 6. Conclusiones generales

<div style="background:#242938;border-left:5px solid #c9a9fb;border-radius:8px;padding:16px 20px;color:#dcdfea;">

<p>🔤 <b>Legibilidad:</b> SQL fue más claro en las agregaciones simples (comparaciones 2 y 3) GROUP BY más una expresión en el SELECT se lee de corrido. DataFrame API ganó en la comparación 4 (window function + CTE): encadenar .groupBy().count().withColumn(rank).filter() es más fácil de seguir paso a paso que anidar dos CTEs.</p>

<p>🧩 <b>Composición:</b> encadenar transformaciones en DataFrame API (.filter().groupBy().agg()) hace visible el orden real de las operaciones; anidar SQL (subqueries, CTEs) obliga a leer de dentro hacia fuera, aunque las CTEs con nombre (WITH counts AS (...)) ayudan a que ese anidamiento se lea casi como pasos.</p>

<p>⚙️ <b>Planes de ejecución:</b> de las 4 comparaciones, <b>3 produjeron un Physical Plan idéntico</b> carácter a carácter. La única diferencia (comparación 3) fue un nodo Project de más en DataFrame API, sin coste extra real (mismo número de Exchange). Cero diferencias de rendimiento entre las dos APIs para consultas equivalentes.</p>

<p>🤔 <b>Cuándo usaría cada una:</b> SQL cuando la consulta es autocontenida y la va a revisar alguien de analytics/BI sin contexto de Python. DataFrame API cuando la lógica se mezcla con código — tests, funciones reutilizables, parámetros que cambian dinámicamente (como el año en la comparación 1, que en DataFrame API es una variable Python y en SQL habría que interpolar el string).</p>

</div>